In [ ]:
# Cell 1 - Install packages if needed
# Run this only once in Jupyter if the packages are not installed.

%pip install pandas sqlalchemy pybigquery google-cloud-bigquery

In [ ]:
# Cell 2 - Imports and display settings

from __future__ import annotations

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 200)

In [ ]:
# Cell 3 - Configuration
# Update these paths/values to match your environment.

import os

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET = os.getenv("BQ_ANALYTICS_DATASET", "olist_analytics")
SERVICE_ACCOUNT_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# SQLAlchemy connection string for BigQuery via pybigquery
CONNECTION_URI = (
    f"bigquery://{PROJECT_ID}/{DATASET}"
    f"?credentials_path={SERVICE_ACCOUNT_PATH}"
)

In [ ]:
# Cell 4 - Create SQLAlchemy engine and test connection

engine = create_engine(CONNECTION_URI)

test_query = "SELECT 1 AS ok"
with engine.connect() as conn:
    test_df = pd.read_sql(text(test_query), conn)

print("Connection test result:")
display(test_df)

In [ ]:
# Cell 5 - Quick table row counts (EDA sanity check)

row_count_sql = f"""
SELECT 'fct_order_items' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.fct_order_items`
UNION ALL
SELECT 'fct_orders' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.fct_orders`
UNION ALL
SELECT 'fct_reviews' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.fct_reviews`
UNION ALL
SELECT 'fct_payments' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.fct_payments`
UNION ALL
SELECT 'dim_products' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.dim_products`
UNION ALL
SELECT 'dim_customers' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.dim_customers`
UNION ALL
SELECT 'dim_sellers' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.dim_sellers`
UNION ALL
SELECT 'dim_dates' AS table_name, COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.dim_dates`
ORDER BY table_name
"""

with engine.connect() as conn:
    df_row_counts = pd.read_sql(text(row_count_sql), conn)

display(df_row_counts)